Accelerator: GPU T4 x2, Internet on, and attach the `kaggle_prepare_data` output under Add Input → Your Work → Notebook Output.

In [ ]:
REPO_URL = "https://github.com/<you>/candidate_reranker.git"
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)     # rerun this cell after every push
K.gpu_info()
env = K.prepare(COMMIT)

In [ ]:
assert K.run(env, "selftest.py",
             "--work", "/kaggle/working",
             "--n_utts", "3",
             "--n_candidates", "5") == 0

In [ ]:
SPLIT = "test-clean"
TAG = f"{SPLIT}-{COMMIT}"

K.run(env, "dump_candidates.py",
      "--source", "librispeech",
      "--path", env.librispeech / SPLIT,
      "--base_model", env.base_model,
      "--adapter", env.adapter,
      "--out", env.results / f"{TAG}.jsonl",
      "--tag", TAG,
      "--n_candidates", 15,
      "--n_steps", 4,
      "--resume")

In [ ]:
K.run(env, "analyze.py",
      env.results / f"{TAG}.jsonl",
      "--json", env.results / f"{TAG}.stats.json")